In [1]:
import pandas as pd

In [2]:
%%capture
!pip install -U dspy pydantic dspy-ai

In [3]:
import dspy

api_key = ""
api_endpoint = "https://<endpoint>.services.ai.azure.com/"

lm = dspy.LM('azure/gpt-4.1', api_key = api_key, api_base=api_endpoint, api_version = '2024-10-21', max_tokens=4000)
dspy.configure(lm=lm)

In [4]:
from typing import List
from typing import Literal
from pydantic import BaseModel

class PatientRecord(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all tokens of toxic habits, if any.
    """
    
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    #Tobacco, Cannabis, Alcohol and Drug - type
    tobacco_habits: list[str] = dspy.OutputField(desc="All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.")
    cannabis_habits: list[str] = dspy.OutputField(desc="All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.")
    alcohol_habits: list[str] = dspy.OutputField(desc="All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.")
    drug_habits: list[str] = dspy.OutputField(desc="All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.")

term_extractor_patient = dspy.ChainOfThought(PatientRecord)

In [5]:
text = """
Mujer de casi 32 años, natural y residente en la zona, en seguimiento en nuestra UCA de Vinaròs desde los 18 años (en 2005); en terapia conmigo desde 2014 (año de mi incorporación a la plaza).
De acuerdo a las notas de la historia clínica de papel y diferentes documentos consultados para la sesión clínica -a menudo desordenados, informes de distinta procedencia y cotejo con apuntes propios-, la paciente se inició en el consumo de tabaco y alcohol a los 12 años, en el cannabis a los 13 (diario desde los 15), en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol), y años más tarde consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox.
Lena dejó los estudios con 16 años, en 2º de la ESO, optando luego por trabajos muy precarios, erráticos, y sobre todo actividades marginales, entre ellas la prostitución hace 2 o 3 años, en Valencia.
Tiene reconocida una PNC (pensión no contributiva) por discapacidad del 73%, de la cual ella siempre se ha administrado el dinero, y hace un año fue inscrita por los Servicios Sociales de la localidad en un curso remunerado de administrativo, donde las condiciones de asistencia eran relativamente exigentes (horarios madrugadores, puntualidad, presencia, exposiciones), y a Lena le costaba bastante cumplir.
Mostraba esfuerzo, también motivada por el incentivo económico, pero había días que no acudía a clase, y ésta era también una de las razones para asistir con más frecuencia y compromiso a las citas médicas y psicológicas, que justificaban la ausencia de clase ese día.
Posteriormente ingresó voluntariamente en un centro de día para rehabilitación de tóxicos, con horarios poco compatibles con el curso de administrativo, sin perder la plaza gracias a una ILT (baja médica).
Respecto al entorno de Lena, la familia se caracteriza por su carencia de estructura.
Los padres de la paciente eran toxicómanos antes y durante su infancia, y ambos ya están fallecidos.
Ella fue acogida por la abuela materna, que también ha sido la persona que ha criado a una hermana 14 años menor, de un padre diferente (éste se encuentra vivo, también era toxicómano, fue presidiario, en la actualidad visita ocasionalmente a la familia, sobre todo a su hija, la hermana de Lena que ahora tiene 18 años).
Por tanto, desde el nacimiento la tutela de la paciente ha estado con los abuelos maternos, que actualmente cuentan con 68 años la mujer y 64 el marido (este hombre puede no ser el abuelo biológico de Lena, según una referencia de un informe de la historia clínica, y es la persona sobre la que actualmente ella deposita más hostilidad y rechazo).
La abuela es quien siempre se encarga de acompañar a Lena a los dispositivos, la ha rescatado en numerosas ocasiones de sitios hostiles y caóticos, ha supervisado muchas veces tratamientos y medicaciones, gestionado citas y visitas… es la principal figura de apego de la paciente, sin duda.
"""
term_extractor_patient(patient_discharge_summary=text)

Prediction(
    reasoning='The discharge summary provides a detailed history of substance use. The patient started "consumo de tabaco y alcohol" at age 12, "en el cannabis a los 13 (diario desde los 15)", and "en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol)", and later "consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox." These are explicit mentions of tobacco, cannabis, alcohol, and various drugs (cocaína, speed, anfetaminas, éxtasis, heroína, drogas). The summary also mentions "rehabilitación de tóxicos" and "toxicómanos" in reference to the patient and her family, which are relevant drug-related terms.',
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína', 'drogas', 'tóxicos', 'toxicómanos']
)

In [6]:
result_filename = 'pred_test_gpt41_lists_examples_per_field_temp0'

In [7]:
df_records = pd.read_csv('final_test_dataset.tsv', sep='\t')

In [8]:
df_records.head()

,filename,text,trigger_annotations,attr_annotations
0,casos_clinicos_cardiologia117,Varón de 70 años. Antecedente de tuberculosis ...,[],[]
1,casos_clinicos_cardiologia483,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...",[],[]
2,casos_clinicos_habtox1,"Dolor mamario y náuseas matutinas, refiere tes...",[],[]
3,casos_clinicos_habtox11,"Se presenta el caso de un hombre de 37 años, c...",[],[]
4,casos_clinicos_habtox114,"Varón de 40 años, procedente de la Amazonía pe...",[],[]


In [9]:
from ast import literal_eval
df_records['trigger_annotations'] = df_records['trigger_annotations'].apply(literal_eval)
df_records['attr_annotations'] = df_records['attr_annotations'].apply(literal_eval)

In [10]:
df_records['response'] = ''
df_records['entities'] = ''
df_records['is_train'] = False
df_records_dev = df_records[~df_records['is_train']]

In [11]:
df_records_dev.shape

(300, 7)

In [12]:
from tqdm import tqdm, tqdm_notebook

for index, row in tqdm(df_records_dev.iterrows(), total=df_records_dev.shape[0]):
    if row['response'] != '':
        continue
    responses = []
    entities = []
    texts = row['text'].split('\n\n')
    for text in texts:
        response = term_extractor_patient(patient_discharge_summary=text) 
        responses.append(response)
        current_entities = []
        for habit in response.tobacco_habits:
            current_entities.append({
                'trigger_type': 'Tobacco',
                'trigger_text': habit
            })
        for habit in response.alcohol_habits:
            current_entities.append({
                'trigger_type': 'Alcohol',
                'trigger_text': habit
            })
        for habit in response.cannabis_habits:
            current_entities.append({
                'trigger_type': 'Cannabis',
                'trigger_text': habit
            })
        for habit in response.drug_habits:
            current_entities.append({
                'trigger_type': 'Drug',
                'trigger_text': habit
            })
        entities.extend(current_entities)
    df_records_dev.at[index, 'response'] = responses
    df_records_dev.at[index, 'entities'] = entities

100%|████████████████████████████████████████████████████████████████████████| 300/300 [00:02<00:00, 100.87it/s]


In [13]:
df_records_dev.head()

,filename,text,trigger_annotations,attr_annotations,response,entities,is_train
0,casos_clinicos_cardiologia117,Varón de 70 años. Antecedente de tuberculosis ...,[],[],"[[reasoning, tobacco_habits, cannabis_habits, ...","[{'trigger_type': 'Tobacco', 'trigger_text': '...",False
1,casos_clinicos_cardiologia483,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...",[],[],"[[reasoning, tobacco_habits, cannabis_habits, ...","[{'trigger_type': 'Tobacco', 'trigger_text': '...",False
2,casos_clinicos_habtox1,"Dolor mamario y náuseas matutinas, refiere tes...",[],[],"[[reasoning, tobacco_habits, cannabis_habits, ...","[{'trigger_type': 'Tobacco', 'trigger_text': '...",False
3,casos_clinicos_habtox11,"Se presenta el caso de un hombre de 37 años, c...",[],[],"[[reasoning, tobacco_habits, cannabis_habits, ...","[{'trigger_type': 'Alcohol', 'trigger_text': '...",False
4,casos_clinicos_habtox114,"Varón de 40 años, procedente de la Amazonía pe...",[],[],"[[reasoning, tobacco_habits, cannabis_habits, ...","[{'trigger_type': 'Alcohol', 'trigger_text': '...",False


In [26]:
df_records_dev.iloc[10]['entities']

[{'trigger_type': 'Alcohol', 'trigger_text': 'alcohol'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'alcoholismo'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'alcoholemia'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'hepatitis alcohólica'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'embriagado'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'coma etílico'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'pancreatitis aguda'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'abstinencia'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'alcohólicas'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'UBE'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'aguardiente'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'vino'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'cañas'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'cervezas'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'coñac'},
 {'trigger_type': 'Alcohol', 'trigger_text': 'combinado'},
 {'trigger_type': 'Alcohol', 'trigger_text':

In [17]:
df_records_dev.to_csv(f'{result_filename}.tsv', sep='\t', index=False)

In [27]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [28]:
import re

entities_list = []

for index, row in df_records_dev.iterrows():
    if row['response'] == '':
        continue
        
    entities = row['entities']
    text = row['text']
    
    entity_list = []
    for ent in entities:
        term = ent[f'trigger_text']
        label = ent['trigger_type']

        entity_list.extend(get_entities(row['filename'], text, term, label))        
        
    entities_list.extend(entity_list)    

In [29]:
import pandas as pd

df_entities_list = pd.DataFrame.from_records(entities_list)
df_entities_list.drop_duplicates(inplace=True)
df_entities_list.head()

,filename,mark,label,off0,off1,span
0,casos_clinicos_cardiologia117,TOX,Tobacco,342,370,Exfumador de 50 paquetes año
1,casos_clinicos_cardiologia483,TOX,Tobacco,80,88,fumadora
2,casos_clinicos_habtox1,TOX,Tobacco,3208,3214,Tabaco
3,casos_clinicos_habtox1,TOX,Alcohol,3216,3223,alcohol
4,casos_clinicos_habtox1,TOX,Alcohol,2927,2943,Enolismo crónico


In [30]:
#'mark'
df_entities_list[['filename','label','off0','off1','span']].to_csv(f'{result_filename}_entities.tsv', sep='\t', index=False)
df_entities_list.shape

(2601, 6)

In [22]:
result_filename

'pred_test_gpt41_lists_examples_per_field_temp0'